# PYNQ-Z2 HDMI 输出验证（720p）

先按 BTN3 结束旧的 LED 示例。HDMI OUT 连接显示器 HDMI 输入，选择相应输入源。

按顺序运行两个代码单元；不要点击 Run All。最后的清理单元等拍照后再运行。
本测试复用官方 PYNQ 3.0.1 BaseOverlay、视频驱动和 VDMA；不需要 HDMI IN。


## 1. 加载官方 Base 并启动显示输出
点击下面代码框，按 Shift + Enter，等待出现启动提示。


In [1]:
from pynq.overlays.base import BaseOverlay
from pynq.lib.video import VideoMode, PIXEL_BGR

base = BaseOverlay("base.bit")
hdmi_out = base.video.hdmi_out
mode = VideoMode(1280, 720, 24, fps=60)
hdmi_out.configure(mode, PIXEL_BGR)
hdmi_out.start()
print("HDMI output started: 1280 x 720, 60 Hz, BGR24")


HDMI output started: 1280 x 720, 60 Hz, BGR24


## 2. 显示彩色带
运行下面代码后，预期从左至右为蓝、绿、红三条彩色带。观察至少 30 秒并拍照；打印提示只表示提交图像，不自动证明显示通过。


In [6]:
frame = hdmi_out.newframe()
width = mode.width
frame[:, :width // 3, :] = [255, 0, 0]
frame[:, width // 3:2 * width // 3, :] = [0, 255, 0]
frame[:, 2 * width // 3:, :] = [0, 0, 255]
hdmi_out.writeframe(frame)
print("Color bars submitted. Check the display: BLUE | GREEN | RED.")


Color bars submitted. Check the display: BLUE | GREEN | RED.


In [2]:
print("Mode:", hdmi_out.mode)
f = hdmi_out.newframe()
print("Shape:", f.shape)
print("Type:", f.dtype, "Strides:", f.strides)
f[:] = [0, 0, 255]
hdmi_out.writeframe(f)
print("Red frame submitted; check the display.")

Mode: VideoMode: width=1280 height=720 bpp=24 fps=60
Shape: (720, 1280, 3)
Type: uint8 Strides: (3840, 3, 1)
Red frame submitted; check the display.


In [3]:
g = hdmi_out.newframe()
h, w, _ = g.shape

g[:, :w // 3, :] = [255, 0, 0]
g[:, w // 3:2 * w // 3, :] = [0, 255, 0]
g[:, 2 * w // 3:, :] = [0, 0, 255]

g[:12, :, :] = 255
g[-12:, :, :] = 255
g[:, :12, :] = 255
g[:, -12:, :] = 255

g[:, w // 2 - 4:w // 2 + 4, :] = 255
g[h // 2 - 4:h // 2 + 4, :, :] = 255

hdmi_out.writeframe(g)
print("Position test submitted.")

Position test submitted.


In [4]:
import time
import json
from pathlib import Path

height, width = hdmi_out.mode.height, hdmi_out.mode.width
submitted = 0

for n in range(100):
    f = hdmi_out.newframe()
    f[:] = [40, 40, 40]
    x = n * (width - 64) // 99
    f[60:height - 60, x:x + 64, :] = [255, 255, 255]
    hdmi_out.writeframe(f)
    submitted += 1
    time.sleep(0.05)

Path("hdmi_framebuffer_100_frames.json").write_text(
    json.dumps({"submitted_frames": submitted,
                "width": width, "height": height,
                "screen_observation": "PENDING_USER_CONFIRMATION"})
)
print("Submitted", submitted, "frames.")

Submitted 100 frames.


## 3. 拍照完成后停止显示

先保留显示器照片和运行输出，再运行下面单元。停止后显示器可能变黑或提示无信号。
如果前面的代码报错，保留报错并暂停，不继续操作其他 Notebook 加载 FPGA。

来源：项目已有 board_validation.ipynb 的 B03 输出流程；官方接口见 [HDMI 驱动](https://github.com/Xilinx/PYNQ/blob/v3.0.1/pynq/lib/video/hierarchies.py) 和 [VDMA 驱动](https://github.com/Xilinx/PYNQ/blob/v3.0.1/pynq/lib/video/dma.py)。官方 HDMI 入门 Notebook 需要输入源，本测试仅使用输出接口。


In [5]:
hdmi_out.close()
print("HDMI output closed.")


HDMI output closed.
